# mmrag-eval Quickstart

This notebook walks through evaluating a multimodal RAG system with all three metrics.

In [ ]:
# Install if needed
# !pip install mmrag-eval

## 1. Create synthetic images for demonstration

In [ ]:
import os
from PIL import Image

os.makedirs('demo_images', exist_ok=True)

colors = {
    'red_diagram.png':   (220, 50,  50),
    'blue_chart.png':    (50,  80,  220),
    'green_table.png':   (50,  180, 80),
    'red_diagram2.png':  (215, 55,  55),   # near-duplicate of red_diagram
}

for fname, color in colors.items():
    img = Image.new('RGB', (224, 224), color=color)
    img.save(f'demo_images/{fname}')
    print(f'Created demo_images/{fname}')

## 2. Build a dataset

In [ ]:
from mmrag_eval.dataset.loader import MMRagSample

samples = [
    MMRagSample(
        query='What does the red diagram show?',
        image_path='demo_images/red_diagram.png',
        reference_answer='A red flowchart illustrating the data processing pipeline.',
        grounding_labels=['demo_images/red_diagram.png'],
    ),
    MMRagSample(
        query='Describe the chart in the document.',
        image_path='demo_images/blue_chart.png',
        reference_answer='A blue bar chart comparing model accuracy across datasets.',
        grounding_labels=['demo_images/blue_chart.png'],
    ),
]

## 3. Simulate retrieval with a custom grounding function

We use a stub grounding function here so no CLIP model download is needed.

In [ ]:
# Stub: returns a fixed plausible score
def stub_grounding(image_path: str, text: str) -> float:
    return 0.78

retrieved_images = [
    # Sample 0: correct image retrieved first, then a near-duplicate
    ['demo_images/red_diagram.png', 'demo_images/red_diagram2.png', 'demo_images/green_table.png'],
    # Sample 1: correct image retrieved first
    ['demo_images/blue_chart.png', 'demo_images/green_table.png'],
]

generated_answers = [
    'The diagram shows a red-coloured processing pipeline flowchart.',
    'The blue bar chart compares accuracy across multiple benchmark datasets.',
]

## 4. Run the full evaluation pipeline

In [ ]:
from mmrag_eval import evaluate
import json

results = evaluate(
    samples=samples,
    retrieved_images=retrieved_images,
    generated_answers=generated_answers,
    k=3,
    grounding_fn=stub_grounding,
)

print('=== Aggregated Scores ===')
print(json.dumps(results['aggregated'], indent=2))

## 5. Inspect individual metrics

In [ ]:
from mmrag_eval.metrics import retrieval_quality, diversity

# Retrieval quality for sample 0
rq = retrieval_quality.score(
    retrieved_images=retrieved_images[0],
    relevant_images=samples[0].grounding_labels,
    k=3,
)
print('Retrieval quality (sample 0):', rq)

# Diversity — note sample 0 has a near-duplicate pair
div = diversity.score(image_paths=retrieved_images[0], threshold=10)
print('Diversity (sample 0):', div)

## 6. Using CLIP (real grounding fidelity)

Remove the `grounding_fn` argument to use CLIP. This downloads `openai/clip-vit-base-patch32` on first call (~600 MB).

```python
results = evaluate(
    samples=samples,
    retrieved_images=retrieved_images,
    generated_answers=generated_answers,
    k=3,
    # grounding_fn omitted → CLIP is used
)
```